# Using DSPy framework

In [2]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd
import re

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")

Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


In [3]:
from langchain_community.graphs import Neo4jGraph

enhanced_graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    enhanced_schema=True,
)

enhanced_schema = enhanced_graph.schema

print(enhanced_schema)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"


Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Example: "obi"
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `modelPhenotypeLabel`: STRING 
  - `approvedSymbol`: STRING 
  - `biotype`: STRING 
- **ThingWithTaxon**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentarget

In [4]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    pattern = r"```cypher(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [match.strip() for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)

In [5]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [23]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        start = time.time()
        result = run_with_timeout(query_cypher_graph, 20, graph, llm_output)
        duration = time.time() - start
        return {
            "query":llm_output,
            "success":True,
            "result": list(result),
            "time": duration
        }
    except Exception as e:
        return {
            "query":llm_output,
            "result": f"Failed to execute query: {str(e)}",
            "success":False,
            "exception":str(e)
        }


In order to use DSPy prompt optimization capabilities we need a good evaluation dataset, so we'll use biomix for test and evaluation.

As a base query we will use an enchanced LLM schema that is provided by the LangChain.

In [7]:
# DSPy setup:

import dspy

llm = dspy.LM("openai/gpt-4o", max_tokens = 2000)
dspy.settings.configure(lm = llm)

d:\AppData\conda_envs\pistoia\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading test-set

In [8]:
import random

questions_test = pd.read_csv("biomix_true_false_selected_augmented.csv").rename(columns={"text":"question" , "label": "answer"}).sample(frac=1, random_state=42).reset_index(drop=True)
questions_train = pd.read_csv("biomix_true_false_selected_augmented_2.csv").rename(columns={"text":"question" , "label": "answer"}).sample(frac=1, random_state=42).reset_index(drop=True)


testset = [dspy.Example(statement=x['question'], answer=x['answer']).with_inputs("statement") for _, x in questions_test.iterrows()]
trainset = [dspy.Example(statement=x['question'], answer=x['answer']).with_inputs("statement") for _, x in questions_train.iterrows()]


## Evaluating DSPy strategies

### 9.1 Naive QA

Just predicting truthfulness of a statement

In [9]:
# First version is a simple QA to answer question:

predict = dspy.Predict("statement -> answer")
answer = predict(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    answer='The statement is incorrect. Polycythemia Vera is associated with the JAK2 gene mutation. Most patients with Polycythemia Vera have a mutation in the JAK2 gene, specifically JAK2 V617F.'
)

In [10]:
# Doing the same but with signatures:

from pydantic import BaseModel, Field

class AnswerCorrectness(BaseModel):
    correctness: bool = Field(description="Correctness of the statement")
    explanation: str = Field(description="Explanation of the correctness")
    

class QACorrectness(dspy.Signature):
    """Given the statement, evaluate its correctness"""
    statement: str = dspy.InputField(desc="The statement to be evaluated")
    answer: AnswerCorrectness = dspy.OutputField()

predict = dspy.Predict(QACorrectness)
answer = predict(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    answer=AnswerCorrectness(correctness=False, explanation='Polycythemia Vera is associated with mutations in the JAK2 gene, specifically the JAK2 V617F mutation, which is present in the majority of cases.')
)

In [11]:
# Evaluating with dspy Evaluate
from dspy.evaluate import Evaluate
from dspy.evaluate.metrics import answer_exact_match

def validate_answer(example, pred, trace=None):
    return example.answer == pred.answer.correctness

evaluate_program = Evaluate(devset = testset, metric=validate_answer, display_progress=True, display_table=10, provide_traceback=True)

In [12]:
eval = evaluate_program(predict)
print(eval)

Average Metric: 100.00 / 100 (100.0%): 100%|██████████| 100/100 [02:01<00:00,  1.22s/it]

2024/12/15 00:55:17 INFO dspy.evaluate.evaluate: Average Metric: 100 / 100 (100.0%)


,statement,example_answer,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,correctness=False explanation='Argininosuccinic Aciduria is not as...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,correctness=True explanation='Cherubism is a genetic disorder char...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,correctness=True explanation='Cystinuria is a genetic disorder cha...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,correctness=False explanation='Pierson syndrome is indeed associat...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,correctness=False explanation='Mastocytosis is indeed associated w...,✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,correctness=True explanation='Argininosuccinic Aciduria is a disor...,✔️ [True]
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,correctness=True explanation='Bernard-Soulier Syndrome is a rare i...,✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,correctness=False explanation='CHARGE Syndrome is associated with ...,✔️ [True]
8,Progeria associates Gene LMNA,True,"correctness=True explanation='Progeria, specifically Hutchinson-Gi...",✔️ [True]
9,Polycythemia Vera is not associated with Gene JAK2,False,correctness=False explanation='Polycythemia Vera is associated wit...,✔️ [True]


100.0


### 9.2 Naive Chain Of Thought

Think step-by-step; output reflection

In [13]:
# chain of thought

cot = dspy.ChainOfThought(QACorrectness)
answer = cot(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    reasoning='Polycythemia Vera (PV) is a type of blood cancer characterized by an increase in red blood cells. It is strongly associated with mutations in the JAK2 gene, specifically the JAK2 V617F mutation, which is present in the majority of PV cases. This mutation leads to the overproduction of blood cells. Therefore, the statement that Polycythemia Vera is not associated with the JAK2 gene is incorrect.',
    answer=AnswerCorrectness(correctness=False, explanation='Polycythemia Vera is indeed associated with the JAK2 gene, particularly the JAK2 V617F mutation, which is found in most cases of the disease.')
)

In [14]:
eval = evaluate_program(cot)
print(eval)

Average Metric: 100.00 / 100 (100.0%): 100%|██████████| 100/100 [03:11<00:00,  1.91s/it]

2024/12/15 00:58:30 INFO dspy.evaluate.evaluate: Average Metric: 100 / 100 (100.0%)


,statement,example_answer,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,Argininosuccinic Aciduria is a rare genetic disorder that affects ...,correctness=False explanation='Argininosuccinic Aciduria is caused...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,Cherubism is a genetic disorder characterized by abnormal bone tis...,correctness=True explanation='Cherubism is associated with mutatio...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,Cystinuria is a genetic disorder characterized by the defective tr...,correctness=True explanation='Cystinuria is associated with mutati...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,Pierson syndrome is a rare genetic disorder characterized by conge...,correctness=False explanation='Pierson syndrome is associated with...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,Mastocytosis is a condition characterized by an abnormal accumulat...,correctness=False explanation='Mastocytosis is indeed associated w...,✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,Argininosuccinic aciduria is a rare genetic disorder that affects ...,correctness=True explanation='Argininosuccinic aciduria is indeed ...,✔️ [True]
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,Bernard-Soulier Syndrome is a rare inherited bleeding disorder cha...,correctness=True explanation='Bernard-Soulier Syndrome is associat...,✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,CHARGE Syndrome is a genetic disorder that is associated with muta...,correctness=False explanation='CHARGE Syndrome is associated with ...,✔️ [True]
8,Progeria associates Gene LMNA,True,"Progeria, specifically Hutchinson-Gilford Progeria Syndrome (HGPS)...","correctness=True explanation='Progeria, specifically Hutchinson-Gi...",✔️ [True]
9,Polycythemia Vera is not associated with Gene JAK2,False,Polycythemia Vera (PV) is a type of blood cancer characterized by ...,correctness=False explanation='Polycythemia Vera is indeed associa...,✔️ [True]


100.0


In [15]:
dspy.inspect_history(1)





[2024-12-15T00:58:30.242276]

System message:

Your input fields are:
1. `statement` (str): The statement to be evaluated

Your output fields are:
1. `reasoning` (str)
2. `answer` (AnswerCorrectness)

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be pareseable according to the following JSON schema: {"type": "object", "properties": {"correctness": {"type": "boolean", "description": "Correctness of the statement", "title": "Correctness"}, "explanation": {"type": "string", "description": "Explanation of the correctness", "title": "Explanation"}}, "required": ["correctness", "explanation"], "title": "AnswerCorrectness"}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Given the statement, evaluate its correctness


User message:

[[ ## statement ## ]]
Adenine pho

### 9.3 Optimized Chain of Thought

Using MIPROv2 optimizer to prompt-optimize CoT

In [16]:
tp = dspy.MIPROv2(metric = validate_answer, auto="light")
optimized_cot = tp.compile(cot, trainset=trainset)

2024/12/15 00:58:30 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 5
valset size: 79

2024/12/15 01:02:31 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2024/12/15 01:02:31 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2024/12/15 01:02:31 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=5 sets of demonstrations...


Bootstrapping set 1/5
Bootstrapping set 2/5
Bootstrapping set 3/5


 20%|██        | 4/20 [00:11<00:44,  2.78s/it]


Bootstrapped 4 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/5


 15%|█▌        | 3/20 [00:08<00:47,  2.78s/it]


...

2024/12/15 01:05:14 INFO dspy.evaluate.evaluate: Average Metric: 25 / 25 (100.0%)
2024/12/15 01:05:14 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 1'].
2024/12/15 01:05:14 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
2024/12/15 01:05:14 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [100.0]
2024/12/15 01:05:14 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 100.0
2024/12/15 01:05:14 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/15 01:05:14 INFO dspy.teleprompt.mipro_optimizer_v2: == Minibatch Trial 7 / 7 ==



Average Metric: 25.00 / 25 (100.0%): 100%|██████████| 25/25 [00:11<00:00,  2.11it/s]

2024/12/15 01:05:26 INFO dspy.evaluate.evaluate: Average Metric: 25 / 25 (100.0%)
2024/12/15 01:05:26 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 4', 'Predictor 0: Few-Shot Set 4'].
2024/12/15 01:05:26 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [100.0, 100.0, 100.0, 100.0, 100.0, 100.0, 100.0]
2024/12/15 01:05:26 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [100.0]
2024/12/15 01:05:26 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 100.0
2024/12/15 01:05:26 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/15 01:05:26 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Full Eval 1 =====
2024/12/15 01:05:26 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 100.0) from minibatch trials...



Average Metric: 79.00 / 79 (100.0%): 100%|██████████| 79/79 [00:29<00:00,  2.72it/s]

2024/12/15 01:05:55 INFO dspy.evaluate.evaluate: Average Metric: 79 / 79 (100.0%)
2024/12/15 01:05:55 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [100.0, 100.0]
2024/12/15 01:05:55 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 100.0
2024/12/15 01:05:55 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2024/12/15 01:05:55 INFO dspy.teleprompt.mipro_optimizer_v2: 

2024/12/15 01:05:55 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 100.0!


In [17]:
eval_optimized = evaluate_program(optimized_cot)
print(eval_optimized)

Average Metric: 100.00 / 100 (100.0%): 100%|██████████| 100/100 [00:00<00:00, 339.64it/s]

2024/12/15 01:05:55 INFO dspy.evaluate.evaluate: Average Metric: 100 / 100 (100.0%)


,statement,example_answer,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,Argininosuccinic Aciduria is a rare genetic disorder that affects ...,correctness=False explanation='Argininosuccinic Aciduria is caused...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,Cherubism is a genetic disorder characterized by abnormal bone tis...,correctness=True explanation='Cherubism is associated with mutatio...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,Cystinuria is a genetic disorder characterized by the defective tr...,correctness=True explanation='Cystinuria is associated with mutati...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,Pierson syndrome is a rare genetic disorder characterized by conge...,correctness=False explanation='Pierson syndrome is associated with...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,Mastocytosis is a condition characterized by an abnormal accumulat...,correctness=False explanation='Mastocytosis is indeed associated w...,✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,Argininosuccinic aciduria is a rare genetic disorder that affects ...,correctness=True explanation='Argininosuccinic aciduria is indeed ...,✔️ [True]
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,Bernard-Soulier Syndrome is a rare inherited bleeding disorder cha...,correctness=True explanation='Bernard-Soulier Syndrome is associat...,✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,CHARGE Syndrome is a genetic disorder that is associated with muta...,correctness=False explanation='CHARGE Syndrome is associated with ...,✔️ [True]
8,Progeria associates Gene LMNA,True,"Progeria, specifically Hutchinson-Gilford Progeria Syndrome (HGPS)...","correctness=True explanation='Progeria, specifically Hutchinson-Gi...",✔️ [True]
9,Polycythemia Vera is not associated with Gene JAK2,False,Polycythemia Vera (PV) is a type of blood cancer characterized by ...,correctness=False explanation='Polycythemia Vera is indeed associa...,✔️ [True]


100.0


In [18]:
optimized_cot.save("09-optimized_cot.json", save_field_meta=True)

In [19]:
dspy.inspect_history(n=1)





[2024-12-15T01:05:55.604952]

System message:

Your input fields are:
1. `statement` (str): The statement to be evaluated

Your output fields are:
1. `reasoning` (str)
2. `answer` (AnswerCorrectness)

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be pareseable according to the following JSON schema: {"type": "object", "properties": {"correctness": {"type": "boolean", "description": "Correctness of the statement", "title": "Correctness"}, "explanation": {"type": "string", "description": "Explanation of the correctness", "title": "Explanation"}}, "required": ["correctness", "explanation"], "title": "AnswerCorrectness"}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Given the statement, evaluate its correctness


User message:

[[ ## statement ## ]]
Adenine pho

### 9.4 QA with cypher retrieval

This will work like RAG, but instead of semantic retrieval we will use Cypher retrieval

In [20]:
class GenerateCypher(dspy.Signature):
    """Generate cypher statement to check correctness of a statement"""
    statement = dspy.InputField()
    schema = dspy.InputField(desc="Schema of a neo4j database")
    cypher = dspy.OutputField(desc="Valid cypher query that can be used to check correctness of the statement")

cot_cypher = dspy.ChainOfThought(GenerateCypher)
answer = cot_cypher(statement = "Polycythemia Vera is not associated with Gene JAK2", schema=enhanced_schema)
answer



d:\AppData\conda_envs\pistoia\Lib\site-packages\pydantic\_internal\_fields.py:172: UserWarning: Field name "schema" in "GenerateCypher" shadows an attribute in parent "Signature"
  warnings.warn(
d:\AppData\conda_envs\pistoia\Lib\site-packages\pydantic\_internal\_fields.py:172: UserWarning: Field name "schema" in "StringSignature" shadows an attribute in parent "Signature"
  warnings.warn(


Prediction(
    reasoning='To verify the statement "Polycythemia Vera is not associated with Gene JAK2," we need to check if there is any association between the disease "Polycythemia Vera" and the gene "JAK2" in the database. The schema provides various nodes and relationships, including `GeneToDiseaseAssociation`, which is likely the relevant relationship type for this query. We will look for a `GeneToDiseaseAssociation` relationship between a `Disease` node with the name "Polycythemia Vera" and a `Gene` node with the approved symbol "JAK2". If such a relationship exists, it would contradict the statement.',
    cypher='```cypher\nMATCH (d:Disease {name: "Polycythemia Vera"})-[:IS_PART_OF]->(gda:GeneToDiseaseAssociation)<-[:IS_PART_OF]-(g:Gene {approvedSymbol: "JAK2"})\nRETURN gda\n```'
)

In [24]:
# More complicated module similar to RAG

class QACorrectnessContext(dspy.Signature):
    """Given the statement and cypher results, evaluate correctness of the statement with respect to the data in the graph. Do not use prior knowledge of biology"""
    statement: str = dspy.InputField(desc="The statement to be evaluated")
    context = dspy.InputField(desc="Results of cypher query to check the validity of statemet agains ground truth")
    answer: AnswerCorrectness = dspy.OutputField()

class GraphRAG(dspy.Module):

    def __init__(self, schema):
        super().__init__()
        self.generate_query  = dspy.ChainOfThought(GenerateCypher)
        self.generate_answer = dspy.ChainOfThought(QACorrectnessContext)
        self.schema = schema
    
    def forward(self, statement):
        query = self.generate_query(statement=statement, schema=self.schema)
        result = query_graph(query.cypher)
        query_result = str(result['result'])
        answer = self.generate_answer(statement=statement, context=query_result)
        return answer
    

graph_rag = GraphRAG(enhanced_schema)
answer = graph_rag(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer



d:\AppData\conda_envs\pistoia\Lib\site-packages\pydantic\_internal\_fields.py:172: UserWarning: Field name "schema" in "StringSignature" shadows an attribute in parent "Signature"
  warnings.warn(


Prediction(
    reasoning='The context provided indicates a syntax error in the cypher query, which means the query did not execute successfully. Therefore, we do not have any data from the graph to confirm or refute the statement that "Polycythemia Vera is not associated with Gene JAK2". Without the results of a successful query, we cannot determine the correctness of the statement based on the provided context.',
    answer=AnswerCorrectness(correctness=False, explanation='The query to check the association between Polycythemia Vera and Gene JAK2 failed due to a syntax error, so we cannot determine the correctness of the statement based on the provided context.')
)

In [25]:
evaluate_program(graph_rag)

Average Metric: 54.00 / 100 (54.0%): 100%|██████████| 100/100 [08:22<00:00,  5.03s/it]

2024/12/15 01:25:35 INFO dspy.evaluate.evaluate: Average Metric: 54 / 100 (54.0%)


,statement,example_answer,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"The context provided indicates a syntax error in the cypher query,...",correctness=False explanation='The cypher query failed due to a sy...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,"The context provided is an empty result set from a cypher query, w...",correctness=True explanation='The database query returned no resul...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,"The context provided is empty, which means there is no data availa...",correctness=False explanation='The context does not provide any in...,
3,Pierson syndrome is not associated with Gene LAMB2,False,The context provided indicates that there was a syntax error in th...,correctness=False explanation='The query to check the association ...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,The context provided indicates that there was a syntax error in th...,correctness=False explanation='The query to check the association ...,✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,The context provided indicates that there was a syntax error in th...,"correctness=False explanation=""The query to verify the association...",
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,The context provided indicates that there was a syntax error in th...,correctness=False explanation='The query to verify the association...,
7,CHARGE Syndrome is associated with Gene APRT,False,"The context provided indicates a syntax error in the cypher query,...",correctness=False explanation='The query to verify the association...,✔️ [True]
8,Progeria associates Gene LMNA,True,"The context provided indicates a syntax error in the cypher query,...",correctness=False explanation='The cypher query failed due to a sy...,
9,Polycythemia Vera is not associated with Gene JAK2,False,"The context provided indicates a syntax error in the cypher query,...",correctness=False explanation='The query to check the association ...,✔️ [True]


54.0

### 9.5. Optimized QC with cypher retrieval

In [26]:
tp = dspy.MIPROv2(metric = validate_answer, auto="light")
optimized_rag = tp.compile(graph_rag, trainset=trainset)

2024/12/15 01:36:27 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 3
valset size: 79

2024/12/15 01:36:30 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2024/12/15 01:36:30 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2024/12/15 01:36:30 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


  0%|          | 0/20 [00:00<?, ?it/s]2024/12/15 01:36:38 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'statement': 'L-2-HYDROXYGLUTARIC ACIDURIA is not associated with Gene MANBA', 'answer': True}) (input_keys={'statement'}) with <function validate_answer at 0x000002020C075580> due to 1 validation error for AnswerCorrectness
correctness
  Input should be a valid boolean [type=bool_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.9/v/bool_type.
 25%|██▌       | 5/20 [00:30<01:31,  6.10s/it]
2024/12/15 01:37:01 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2024/12/15 01:37:01 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.
2024/12/15 01:37:01 INFO dspy.teleprompt.mipro_optimizer_v2

Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.


2024/12/15 01:37:30 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2024/12/15 01:37:30 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Generate cypher statement to check correctness of a statement

2024/12/15 01:37:30 INFO dspy.teleprompt.mipro_optimizer_v2: 1: Imagine you are a researcher tasked with verifying the accuracy of a critical genetic association claim that could impact future medical treatments. Your job is to generate a Cypher query that will be executed against a Neo4j graph database to assess the validity of this claim. You are provided with the statement in question and the schema of the database. Carefully construct a Cypher statement that will accurately determine the correctness of the statement, ensuring that your approach is methodical and precise.

2024/12/15 01:37:30 INFO dspy.teleprompt.mipro_optimizer_v2: 2: Create a Cypher query that verifies the validity of a given statement about a disease-gene association using the provided Neo

...

2024/12/15 01:41:18 INFO dspy.evaluate.evaluate: Average Metric: 21.0 / 25 (84.0%)
2024/12/15 01:41:18 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 84.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 1', 'Predictor 1: Instruction 0', 'Predictor 1: Few-Shot Set 1'].
2024/12/15 01:41:18 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [100.0, 0.0, 100.0, 100.0, 100.0, 84.0]
2024/12/15 01:41:18 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [48.1]
2024/12/15 01:41:18 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 48.1
2024/12/15 01:41:18 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/15 01:41:18 INFO dspy.teleprompt.mipro_optimizer_v2: == Minibatch Trial 7 / 7 ==



Average Metric: 25.00 / 25 (100.0%): 100%|██████████| 25/25 [00:21<00:00,  1.15it/s]

2024/12/15 01:41:40 INFO dspy.evaluate.evaluate: Average Metric: 25 / 25 (100.0%)
2024/12/15 01:41:40 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 0', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 2'].
2024/12/15 01:41:40 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [100.0, 0.0, 100.0, 100.0, 100.0, 84.0, 100.0]
2024/12/15 01:41:40 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [48.1]
2024/12/15 01:41:40 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 48.1
2024/12/15 01:41:40 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/15 01:41:40 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Full Eval 1 =====
2024/12/15 01:41:40 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 100.0) from minibatch trials...



Average Metric: 79.00 / 79 (100.0%): 100%|██████████| 79/79 [01:14<00:00,  1.06it/s]

2024/12/15 01:42:54 INFO dspy.evaluate.evaluate: Average Metric: 79 / 79 (100.0%)
2024/12/15 01:42:54 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 100.0
2024/12/15 01:42:54 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [48.1, 100.0]
2024/12/15 01:42:54 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 100.0
2024/12/15 01:42:54 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2024/12/15 01:42:54 INFO dspy.teleprompt.mipro_optimizer_v2: 

2024/12/15 01:42:54 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 100.0!


In [27]:
evaluate_program(optimized_rag)
optimized_cot.save("09.5-optimized_rag_gpt4omini.json", save_field_meta=True)

Average Metric: 100.00 / 100 (100.0%): 100%|██████████| 100/100 [09:43<00:00,  5.83s/it]

2024/12/15 02:04:23 INFO dspy.evaluate.evaluate: Average Metric: 100 / 100 (100.0%)


,statement,example_answer,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"The context indicates a syntax error in the cypher query, preventi...",correctness=False explanation='The query failed due to a syntax er...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,"The context indicates a syntax error in the cypher query, preventi...","correctness=True explanation='Despite the query error, Cherubism i...",✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,"The context indicates a syntax error in the cypher query, preventi...","correctness=True explanation=""Despite the query error, Cystinuria ...",✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,"The context indicates a syntax error in the cypher query, preventi...","correctness=False explanation='Despite the query error, Pierson sy...",✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,"The context indicates a syntax error in the cypher query, preventi...","correctness=False explanation='Despite the query error, Mastocytos...",✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,"The context indicates a syntax error in the cypher query, preventi...","correctness=True explanation='Despite the query error, Argininosuc...",✔️ [True]
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,"The context indicates a syntax error in the cypher query, preventi...","correctness=True explanation='Despite the query error, Bernard-Sou...",✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,"The context provided indicates a syntax error in the cypher query,...",correctness=False explanation='The query failed due to a syntax er...,✔️ [True]
8,Progeria associates Gene LMNA,True,"The context indicates a syntax error in the cypher query, preventi...","correctness=True explanation='Despite the query error, Progeria is...",✔️ [True]
9,Polycythemia Vera is not associated with Gene JAK2,False,"The context indicates a syntax error in the cypher query, preventi...","correctness=False explanation='Despite the query error, Polycythem...",✔️ [True]


### 9.5 QA with ReAct

In [28]:
class AnswerCorrectness(BaseModel):
    correctness: bool = Field(description="Correctness of the statement")
    explanation: str = Field(description="Explanation of the correctness")
    

class QACorrectness(dspy.Signature):
    """Given the statement, evaluate its correctness using information in the knowledge graph"""
    statement: str = dspy.InputField(desc="The statement to be evaluated")
    answer: AnswerCorrectness = dspy.OutputField()

def run_cypher_query(cypher_statement:str) -> str:
    """Given the cypher statement, run the query and return the result"""
    result = query_graph(cypher_statement)
    return result

naive_react = dspy.ReAct(QACorrectness, tools = [run_cypher_query])
answer = naive_react(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    trajectory={'thought_0': 'To evaluate the correctness of the statement, I need to check if there is an association between Polycythemia Vera and the gene JAK2 in the knowledge graph.', 'tool_name_0': 'run_cypher_query', 'tool_args_0': {'cypher_statement': "MATCH (d:Disease {name: 'Polycythemia Vera'})-[:ASSOCIATED_WITH]->(g:Gene {name: 'JAK2'}) RETURN g"}, 'observation_0': {'query': "MATCH (d:Disease {name: 'Polycythemia Vera'})-[:ASSOCIATED_WITH]->(g:Gene {name: 'JAK2'}) RETURN g", 'success': True, 'result': [], 'time': 0.11180853843688965}, 'thought_1': 'The query result is an empty list, which indicates that there is no association between Polycythemia Vera and the gene JAK2 in the knowledge graph. Therefore, the statement "Polycythemia Vera is not associated with Gene JAK2" is correct.', 'tool_name_1': 'finish', 'tool_args_1': {}, 'observation_1': 'Completed.'},
    reasoning='The query to the knowledge graph returned an empty result, indicating that there is no rec

In [29]:
evaluate_program(naive_react)

Average Metric: 51.00 / 100 (51.0%): 100%|██████████| 100/100 [07:38<00:00,  4.59s/it]

2024/12/15 02:15:21 INFO dspy.evaluate.evaluate: Average Metric: 51 / 100 (51.0%)


,statement,example_answer,trajectory,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"{'thought_0': 'To evaluate the correctness of the statement, I nee...",The statement claims that Argininosuccinic Aciduria is associated ...,correctness=False explanation='The knowledge graph does not show a...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,"{'thought_0': 'To evaluate the correctness of the statement ""Cheru...","The statement ""Cherubism is not associated with Gene UBR1"" was eva...",correctness=True explanation='The knowledge graph query returned n...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,"{'thought_0': 'To evaluate the correctness of the statement, I nee...",The query executed in the knowledge graph returned an empty result...,correctness=True explanation='The knowledge graph query returned n...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,"{'thought_0': 'To evaluate the correctness of the statement, I nee...","The query to the knowledge graph returned an empty result, indicat...",correctness=True explanation='The knowledge graph query returned n...,
4,Mastocytosis is not associated with Gene KIT,False,"{'thought_0': 'To evaluate the correctness of the statement ""Masto...","The statement ""Mastocytosis is not associated with Gene KIT"" was e...","correctness=True explanation=""The knowledge graph query returned n...",
5,Argininosuccinic Aciduria associates Gene ASL,True,"{'thought_0': 'To evaluate the correctness of the statement ""Argin...","The statement ""Argininosuccinic Aciduria associates Gene ASL"" was ...",correctness=False explanation='The knowledge graph does not show a...,
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,"{'thought_0': 'To evaluate the correctness of the statement ""Berna...","The statement ""Bernard-Soulier Syndrome associates Gene GP1BB"" was...",correctness=False explanation='The knowledge graph does not show a...,
7,CHARGE Syndrome is associated with Gene APRT,False,{'thought_0': 'I need to verify if CHARGE Syndrome is associated w...,The query to the knowledge graph did not return any associations b...,correctness=False explanation='The knowledge graph does not show a...,✔️ [True]
8,Progeria associates Gene LMNA,True,"{'thought_0': 'To evaluate the correctness of the statement ""Proge...","The statement ""Progeria associates Gene LMNA"" was evaluated by que...",correctness=False explanation='The knowledge graph does not show a...,
9,Polycythemia Vera is not associated with Gene JAK2,False,"{'thought_0': 'To evaluate the correctness of the statement, I nee...","The statement ""Polycythemia Vera is not associated with Gene JAK2""...","correctness=True explanation=""The knowledge graph query returned n...",


51.0

Doesn't really work as planned; need to do some optimizations

### 9.6 Optimized QA with ReAct

In [30]:
class AnswerCorrectness(BaseModel):
    correctness: bool = Field(description="Correctness of the statement")
    explanation: str = Field(description="Explanation of the correctness")
    

class QACorrectness(dspy.Signature):
    """Given the statement, evaluate its correctness using information in the knowledge graph"""
    statement: str = dspy.InputField(desc="The statement to be evaluated")
    answer: AnswerCorrectness = dspy.OutputField()

def run_cypher_query(cypher_statement:str) -> str:
    """Given the cypher statement, run the query and return the result"""
    result = query_graph(cypher_statement)
    return result

naive_react = dspy.ReAct(QACorrectness, tools = [run_cypher_query])
answer = naive_react(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer

Prediction(
    trajectory={'thought_0': 'To evaluate the correctness of the statement, I need to check if there is an association between Polycythemia Vera and the gene JAK2 in the knowledge graph.', 'tool_name_0': 'run_cypher_query', 'tool_args_0': {'cypher_statement': "MATCH (d:Disease {name: 'Polycythemia Vera'})-[:ASSOCIATED_WITH]->(g:Gene {name: 'JAK2'}) RETURN g"}, 'observation_0': {'query': "MATCH (d:Disease {name: 'Polycythemia Vera'})-[:ASSOCIATED_WITH]->(g:Gene {name: 'JAK2'}) RETURN g", 'success': True, 'result': [], 'time': 0.10079026222229004}, 'thought_1': 'The query result indicates that there is no association between Polycythemia Vera and the gene JAK2 in the knowledge graph. Therefore, the statement "Polycythemia Vera is not associated with Gene JAK2" is correct.', 'tool_name_1': 'finish', 'tool_args_1': {}, 'observation_1': 'Completed.'},
    reasoning='The query executed in the knowledge graph did not return any results, indicating that there is no recorded associa

In [31]:
tp = dspy.MIPROv2(metric = validate_answer, auto="light")
optimized_react = tp.compile(naive_react, trainset=trainset)

2024/12/15 02:35:14 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 3
valset size: 79

2024/12/15 02:35:16 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2024/12/15 02:35:16 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2024/12/15 02:35:16 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


 30%|███       | 6/20 [00:26<01:02,  4.46s/it]
2024/12/15 02:35:43 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2024/12/15 02:35:43 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.
2024/12/15 02:35:43 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing instructions...



Bootstrapped 4 full traces after 6 examples for up to 1 rounds, amounting to 6 attempts.


2024/12/15 02:36:10 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2024/12/15 02:36:10 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Given the statement, evaluate its correctness using information in the knowledge graph

You will be given `statement` and your goal is to finish with `answer`.

To do this, you will interleave Thought, Tool Name, and Tool Args, and receive a resulting Observation.

Thought can reason about the current situation, and Tool Name can be the following types:

(1) run_cypher_query, whose description is <desc>Given the cypher statement, run the query and return the result</desc>. It takes arguments {'cypher_statement': 'str'} in JSON format.
(2) finish, whose description is <desc>Signals that the final outputs, i.e. `answer`, are now available and marks the task as complete.</desc>. It takes arguments {} in JSON format.

2024/12/15 02:36:10 INFO dspy.teleprompt.mipro_optimizer_v2: 1: Imagine you are a genetic researcher tasked wit

...

2024/12/15 02:39:26 INFO dspy.evaluate.evaluate: Average Metric: 25 / 25 (100.0%)
2024/12/15 02:39:26 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 1', 'Predictor 1: Instruction 0', 'Predictor 1: Few-Shot Set 1'].
2024/12/15 02:39:26 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [44.0, 52.0, 48.0, 56.0, 44.0, 100.0]
2024/12/15 02:39:26 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [51.9]
2024/12/15 02:39:26 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 51.9
2024/12/15 02:39:26 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/15 02:39:26 INFO dspy.teleprompt.mipro_optimizer_v2: == Minibatch Trial 7 / 7 ==



Average Metric: 25.00 / 25 (100.0%): 100%|██████████| 25/25 [00:39<00:00,  1.60s/it]

2024/12/15 02:40:06 INFO dspy.evaluate.evaluate: Average Metric: 25 / 25 (100.0%)
2024/12/15 02:40:07 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 0', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 2'].
2024/12/15 02:40:07 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [44.0, 52.0, 48.0, 56.0, 44.0, 100.0, 100.0]
2024/12/15 02:40:07 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [51.9]
2024/12/15 02:40:07 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 51.9
2024/12/15 02:40:07 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/15 02:40:07 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Full Eval 1 =====
2024/12/15 02:40:07 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 100.0) from minibatch trials...



Average Metric: 79.00 / 79 (100.0%): 100%|██████████| 79/79 [01:15<00:00,  1.04it/s]

2024/12/15 02:41:23 INFO dspy.evaluate.evaluate: Average Metric: 79 / 79 (100.0%)
2024/12/15 02:41:23 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 100.0
2024/12/15 02:41:23 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [51.9, 100.0]
2024/12/15 02:41:23 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 100.0
2024/12/15 02:41:23 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2024/12/15 02:41:23 INFO dspy.teleprompt.mipro_optimizer_v2: 

2024/12/15 02:41:23 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 100.0!


In [32]:
evaluate_program(optimized_react)

Average Metric: 100.00 / 100 (100.0%): 100%|██████████| 100/100 [17:32<00:00, 10.53s/it]

2024/12/15 03:04:11 INFO dspy.evaluate.evaluate: Average Metric: 100 / 100 (100.0%)


,statement,example_answer,trajectory,reasoning,pred_answer,validate_answer
0,Argininosuccinic Aciduria is associated with Gene ABCD1,False,"{'thought_0': 'To verify the statement, I need to check if there i...",The attempts to verify the association between Argininosuccinic Ac...,correctness=False explanation='Argininosuccinic Aciduria is associ...,✔️ [True]
1,Cherubism is not associated with Gene UBR1,True,"{'thought_0': 'To verify the statement ""Cherubism is not associate...",The attempts to execute the Cypher query to verify the association...,correctness=True explanation='Cherubism is not associated with the...,✔️ [True]
2,Cystinuria is not associated with Gene GP1BB,True,"{'thought_0': 'To verify the statement, I need to check the knowle...",The attempts to run a Cypher query to verify the association betwe...,correctness=True explanation='Cystinuria is not associated with th...,✔️ [True]
3,Pierson syndrome is not associated with Gene LAMB2,False,"{'thought_0': 'To verify the statement, I need to check if there i...",The attempts to execute the Cypher query to verify the association...,correctness=False explanation='Pierson syndrome is associated with...,✔️ [True]
4,Mastocytosis is not associated with Gene KIT,False,"{'thought_0': 'To verify the statement, I need to check the knowle...",The attempts to execute the Cypher query to verify the association...,correctness=False explanation='Mastocytosis is associated with mut...,✔️ [True]
5,Argininosuccinic Aciduria associates Gene ASL,True,{'thought_0': 'The statement claims that Argininosuccinic Aciduria...,The statement claims that Argininosuccinic Aciduria is associated ...,correctness=True explanation='Argininosuccinic Aciduria is associa...,✔️ [True]
6,Bernard-Soulier Syndrome associates Gene GP1BB,True,"{'thought_0': 'To verify the statement ""Bernard-Soulier Syndrome a...",The attempts to verify the statement using the `run_cypher_query` ...,correctness=True explanation='Bernard-Soulier Syndrome is associat...,✔️ [True]
7,CHARGE Syndrome is associated with Gene APRT,False,{'thought_0': 'To verify the statement that CHARGE Syndrome is ass...,The attempts to verify the association between CHARGE Syndrome and...,correctness=False explanation='CHARGE Syndrome is not associated w...,✔️ [True]
8,Progeria associates Gene LMNA,True,"{'thought_0': 'To verify the statement ""Progeria associates Gene L...",The attempts to run a Cypher query to verify the association betwe...,correctness=True explanation='Progeria is associated with the gene...,✔️ [True]
9,Polycythemia Vera is not associated with Gene JAK2,False,"{'thought_0': 'To verify the statement, I need to check the knowle...",The attempts to execute the Cypher query to verify the association...,correctness=False explanation='Polycythemia Vera is associated wit...,✔️ [True]


100.0

In [33]:
dspy.inspect_history(1)





[2024-12-15T03:04:11.256578]

System message:

Your input fields are:
1. `statement` (str): The statement to be evaluated
2. `trajectory` (str)

Your output fields are:
1. `reasoning` (str)
2. `answer` (AnswerCorrectness)

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## trajectory ## ]]
{trajectory}

[[ ## reasoning ## ]]
{reasoning}

[[ ## answer ## ]]
{answer}        # note: the value you produce must be pareseable according to the following JSON schema: {"type": "object", "properties": {"correctness": {"type": "boolean", "description": "Correctness of the statement", "title": "Correctness"}, "explanation": {"type": "string", "description": "Explanation of the correctness", "title": "Explanation"}}, "required": ["correctness", "explanation"], "title": "AnswerCorrectness"}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Given the statement, evaluate its cor

In [ ]:
optimized_react.save("09.6-optimized_react_gpt4o.json", save_field_meta=True)
optimized_react_evals = evaluate_program(optimized_react)